# Notebook for finding sigmoidal transitions in behaviour and dopamine

In [ ]:
import sys
from pathlib import Path

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from model_fit_helpers import (
    fit_continuous_sigmoid_transitions_by_id,
    fit_logistic_transitions_by_id,
    build_realigned_trials,
    fit_curve_series,
)

In [ ]:
DATAFOLDER = ROOT / "data"
FIGSFOLDER = ROOT / "results"
savefigs = False

with open(DATAFOLDER / "assembled_data.pickle", "rb") as f:
    assembled = dill.load(f)

x_array = assembled["x_array"].copy()
print(f"Loaded x_array with {len(x_array)} rows and {x_array.shape[1]} columns")
print(f"Columns include: {', '.join(sorted(x_array.columns)[:12])} ...")

In [ ]:
def get_transition_subset(df, condition="deplete", infusion="45NaCl"):
    return (
        df.query("condition == @condition & infusiontype == @infusion")
        .sort_values(["id", "trial"])
        .copy()
    )


def binarize_median_balance(values, low=-0.7, high=0.7):
    arr = np.asarray(values, dtype=float)
    out = np.full(arr.shape, np.nan, dtype=float)
    out[arr >= high] = 1.0
    out[arr <= low] = 0.0
    return out


def quantize_median_balance(values, low=-0.7, high=0.7):
    arr = np.asarray(values, dtype=float)
    out = np.zeros(arr.shape, dtype=float)
    out[arr >= high] = 1.0
    out[arr <= low] = -1.0
    return out


def summarize_transition_fits(fits_df, id_col="id"):
    if fits_df is None or fits_df.empty:
        return pd.DataFrame()

    summary = fits_df.copy()
    if "x0_orig" in summary.columns:
        summary["x0_orig"] = summary["x0_orig"].round(2)
    if "k" in summary.columns:
        summary["k"] = summary["k"].round(3)

    cols = [c for c in [id_col, "model", "x0_orig", "k", "r_squared", "is_valid", "note"] if c in summary.columns]
    return summary[cols].sort_values(id_col).reset_index(drop=True)

In [ ]:
def plot_subject_fit_traces(fit_traces, y_label, title_prefix, ncols=4, figsize_per_panel=(3.0, 2.4)):
    if not fit_traces:
        print("No fit traces to plot.")
        return None

    subject_ids = sorted(fit_traces.keys())
    n = len(subject_ids)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
        squeeze=False,
    )

    for idx, subject_id in enumerate(subject_ids):
        ax = axes[idx // ncols][idx % ncols]
        trace = fit_traces[subject_id]

        x = trace.get("x", trace.get("x_fit"))
        y = trace.get("y", trace.get("y_fit_raw"))
        y_hat = trace.get("y_hat", trace.get("y_hat_raw"))

        if x is None or y is None:
            ax.axis("off")
            continue

        ax.scatter(x, y, s=16, alpha=0.6, color="#4C5B61", label="data")
        if y_hat is not None:
            ax.plot(x, y_hat, color="#D1495B", lw=1.8, label=trace.get("model", "fit"))
        if np.isfinite(trace.get("x0_orig", np.nan)):
            ax.axvline(trace["x0_orig"], ls="--", lw=1, color="#2F6690", alpha=0.7)

        ax.set_title(str(subject_id), fontsize=9)
        ax.set_xlabel("Trial")
        ax.set_ylabel(y_label)
        sns.despine(ax=ax, offset=3)

    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right", frameon=False)
    fig.suptitle(title_prefix, y=1.02)
    fig.tight_layout()
    return fig


def _keep_complete_aligned_trials(subset, align_col, id_col="id"):
    if subset.empty:
        return subset

    work = subset.copy()
    work["_trial_aligned"] = work[align_col].round().astype(int)

    bounds = work.groupby(id_col)["_trial_aligned"].agg(["min", "max"])
    common_min = int(bounds["min"].max())
    common_max = int(bounds["max"].min())
    if common_min > common_max:
        return work.iloc[0:0].copy()

    expected = np.arange(common_min, common_max + 1, dtype=int)
    work = work.query("_trial_aligned >= @common_min and _trial_aligned <= @common_max").copy()

    valid_ids = []
    for rat_id, rat_df in work.groupby(id_col):
        observed = np.sort(rat_df["_trial_aligned"].unique())
        if np.array_equal(observed, expected):
            valid_ids.append(rat_id)

    return work[work[id_col].isin(valid_ids)].copy()


def _sigmoid_eval(x, params):
    L, k, x0, b = [float(v) for v in params]
    z = np.clip(-k * (x - x0), -60, 60)
    return L / (1 + np.exp(z)) + b


def plot_realignment_overlay(df, align_col, y_cols=("simba_median_balance", "auc_snips"), condition="deplete", infusion="45NaCl"):
    subset = df.query("condition == @condition & infusiontype == @infusion").dropna(subset=[align_col]).copy()
    subset = _keep_complete_aligned_trials(subset, align_col=align_col, id_col="id")

    if subset.empty:
        print(f"No complete aligned rows available for {align_col}")
        return None

    subset["trial_aligned"] = subset[align_col].round().astype(int)
    print(f"Using {subset['id'].nunique()} rats with complete aligned trial windows ({subset['trial_aligned'].min()} to {subset['trial_aligned'].max()})")

    fig, axes = plt.subplots(1, len(y_cols), figsize=(4.5 * len(y_cols), 3.2), sharex=True)
    if len(y_cols) == 1:
        axes = [axes]

    for ax, y_col in zip(axes, y_cols):
        grouped = subset.groupby("trial_aligned")[y_col].agg(["mean", "sem"]).reset_index()
        ax.plot(grouped["trial_aligned"], grouped["mean"], color="#D1495B", lw=2, label="mean")
        ax.fill_between(
            grouped["trial_aligned"],
            grouped["mean"] - grouped["sem"],
            grouped["mean"] + grouped["sem"],
            color="#D1495B",
            alpha=0.25,
        )

        x = grouped["trial_aligned"].to_numpy(dtype=float)
        y = grouped["mean"].to_numpy(dtype=float)
        sig_fit = fit_curve_series(x, y, model_name="sigmoidal", maxfev=30000)
        if sig_fit["success"] and np.all(np.isfinite(sig_fit["params"])):
            x_fit = np.linspace(x.min(), x.max(), 200)
            y_fit = _sigmoid_eval(x_fit, sig_fit["params"])
            k_val = float(sig_fit["params"][1])
            ax.plot(x_fit, y_fit, color="#2F6690", lw=2, ls="--", label="sigmoid fit")
            ax.text(
                0.02,
                0.95,
                f"k={k_val:.3f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=9,
                color="#2F6690",
            )
            print(f"{align_col} | {y_col}: sigmoid k={k_val:.6f}")
        else:
            print(f"{align_col} | {y_col}: sigmoid fit failed")

        ax.axvline(0, color="#2F6690", ls="--", lw=1)
        ax.axhline(0, color="black", ls=":", lw=1, alpha=0.6)
        ax.set_title(y_col)
        ax.set_xlabel("Trials relative to transition")
        ax.set_ylabel("Mean ± SEM")
        sns.despine(ax=ax, offset=3)

    fig.suptitle(f"Realignment using {align_col}", y=1.03)
    fig.tight_layout()
    return fig

### Raw median balance sigmoids

In [ ]:
df_dep45 = get_transition_subset(x_array)

fits_raw, traces_raw = fit_continuous_sigmoid_transitions_by_id(
    df_dep45,
    signal_col="simba_median_balance",
    maxfev=30000,
    require_negative_k=True,
    min_x0=0,
    max_x0=50,
)

print(f"Raw simba_median_balance fits: {len(fits_raw)} valid subjects")
display(summarize_transition_fits(fits_raw))

fig_raw = plot_subject_fit_traces(
    traces_raw,
    y_label="simba_median_balance",
    title_prefix="Raw simba_median_balance sigmoid fits",
)
if savefigs and fig_raw is not None:
    fig_raw.savefig(FIGSFOLDER / "sigmoid_raw_simba_median_balance_subjects.png", dpi=300)

In [ ]:
x_raw_aligned = x_array.copy()
x_raw_aligned["realigned_trials_raw"] = build_realigned_trials(
    x_raw_aligned,
    fits_raw,
    output_col="realigned_trials_raw",
)
x_raw_aligned["trial_aligned"] = x_raw_aligned["realigned_trials_raw"]

fig_raw_align = plot_realignment_overlay(
    x_raw_aligned,
    align_col="realigned_trials_raw",
    y_cols=("simba_median_balance", "auc_snips"),
)
if savefigs and fig_raw_align is not None:
    fig_raw_align.savefig(FIGSFOLDER / "realignment_raw_simba_median_balance.png", dpi=300)

### Binarized median balance sigmoids
1 is median balance > 0.7, 0 is median balance <-0.7, values between -0.7 and 0.7 are ignored/not fitted

In [ ]:
df_dep45 = get_transition_subset(x_array)

fits_bin, traces_bin = fit_logistic_transitions_by_id(
    df_dep45,
    signal_col="simba_median_balance",
    value_transform=lambda values: binarize_median_balance(values, low=-0.7, high=0.7),
    direction="decreasing",
    maxfev=60000,
    min_x0=0,
    max_x0=50,
)

print(f"Binarized simba_median_balance logistic fits: {len(fits_bin)} valid subjects")
display(summarize_transition_fits(fits_bin))

fig_bin = plot_subject_fit_traces(
    traces_bin,
    y_label="binarized simba_median_balance",
    title_prefix="Binarized simba_median_balance logistic fits",
)
if savefigs and fig_bin is not None:
    fig_bin.savefig(FIGSFOLDER / "logistic_binarized_simba_median_balance_subjects.png", dpi=300)

In [ ]:
x_bin_aligned = x_array.copy()
x_bin_aligned["realigned_trials_bin"] = build_realigned_trials(
    x_bin_aligned,
    fits_bin,
    output_col="realigned_trials_bin",
)
x_bin_aligned["trial_aligned"] = x_bin_aligned["realigned_trials_bin"]

fig_bin_align = plot_realignment_overlay(
    x_bin_aligned,
    align_col="realigned_trials_bin",
    y_cols=("simba_median_balance", "auc_snips"),
)
if savefigs and fig_bin_align is not None:
    fig_bin_align.savefig(FIGSFOLDER / "realignment_binarized_simba_median_balance.png", dpi=300)

### Quantized median balance (-1, 0, and +1)
1 is median balance > 0.7, -1 is median balance <-0.7, values between -0.7 and 0.7 are 0

In [ ]:
df_dep45_quant = get_transition_subset(x_array).assign(
    simba_median_balance_quantized=lambda frame: quantize_median_balance(frame["simba_median_balance"].to_numpy(), low=-0.7, high=0.7)
)

fits_quant, traces_quant = fit_continuous_sigmoid_transitions_by_id(
    df_dep45_quant,
    signal_col="simba_median_balance_quantized",
    maxfev=30000,
    require_negative_k=True,
    min_x0=0,
    max_x0=50,
)

print(f"Quantized simba_median_balance sigmoid fits: {len(fits_quant)} valid subjects")
display(summarize_transition_fits(fits_quant))

fig_quant = plot_subject_fit_traces(
    traces_quant,
    y_label="quantized simba_median_balance",
    title_prefix="Quantized (-1/0/1) simba_median_balance sigmoid fits",
)
if savefigs and fig_quant is not None:
    fig_quant.savefig(FIGSFOLDER / "sigmoid_quantized_simba_median_balance_subjects.png", dpi=300)

In [ ]:
x_quant_aligned = x_array.copy()
x_quant_aligned["realigned_trials_quant"] = build_realigned_trials(
    x_quant_aligned,
    fits_quant,
    output_col="realigned_trials_quant",
)
x_quant_aligned["trial_aligned"] = x_quant_aligned["realigned_trials_quant"]

fig_quant_align = plot_realignment_overlay(
    x_quant_aligned,
    align_col="realigned_trials_quant",
    y_cols=("simba_median_balance", "auc_snips"),
)
if savefigs and fig_quant_align is not None:
    fig_quant_align.savefig(FIGSFOLDER / "realignment_quantized_simba_median_balance.png", dpi=300)

In [ ]:
transition_comparison = (
    fits_raw[["id", "x0_orig"]]
    .rename(columns={"x0_orig": "x0_raw_sigmoid"})
    .merge(
        fits_bin[["id", "x0_orig"]].rename(columns={"x0_orig": "x0_binarized_logistic"}),
        on="id",
        how="outer",
    )
    .merge(
        fits_quant[["id", "x0_orig"]].rename(columns={"x0_orig": "x0_quantized_sigmoid"}),
        on="id",
        how="outer",
    )
    .sort_values("id")
)

display(transition_comparison.round(2))

if not transition_comparison.empty:
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    for col, color in [
        ("x0_raw_sigmoid", "#2F6690"),
        ("x0_binarized_logistic", "#D1495B"),
        ("x0_quantized_sigmoid", "#EDAE49"),
    ]:
        vals = transition_comparison[col].dropna()
        ax.scatter(vals, np.repeat(col, len(vals)), alpha=0.8, color=color)

    ax.set_xlabel("Transition trial (x0)")
    ax.set_ylabel("Fit type")
    sns.despine(ax=ax, offset=3)
    fig.tight_layout()